In [0]:
# Store the values from the text input widgets into variables
my_catalog = dbutils.widgets.get("catalog")
my_schema = dbutils.widgets.get("schema")
cust_segment = dbutils.widgets.get("segment")

# Set path to your volume
my_volume_path = f"/Volumes/{my_catalog}/{my_schema}/retail_data"

# display the variables
print(f"my_catalog: {my_catalog}")
print(f"my_schema: {my_schema}")
print(f"my_volume_path: {my_volume_path}")
print(f"cust_segment: {cust_segment}")

In [0]:
# -- 1. Drop the table if it exists
spark.sql(f"DROP TABLE IF EXISTS {my_catalog}.{my_schema}.tb_customer_orders_by_{cust_segment}_segment_gold")

# -- 2. Create the table
query = f"""
CREATE TABLE {my_catalog}.{my_schema}.tb_customer_orders_by_{cust_segment}_segment_gold
USING DELTA
AS
SELECT
   c.customer_id,
  concat(c.first_name,' ',c.last_name) as customer_name,
  c.email,
  c.country,
  c.segment,
  c.is_active,
  o.order_id,
  o.order_date,
  o.status as order_status,
  o.payment_method,
  o.shipping_fee,
  o.order_total
FROM
  {my_catalog}.{my_schema}.tb_customers_bronze as c
  INNER JOIN {my_catalog}.{my_schema}.tb_orders_bronze as o
    ON c.customer_id = o.customer_id
where lower(c.segment) = '{cust_segment}'
    ;
"""

print(query)

# -- 3. Execute the query
spark.sql(query)